# Low-Rank And Randomized Attention Approximations

This notebook tests when attention-like matrices are actually compressible. The through-line is simple: low-rank structure can be real on smooth patterns, but a globally small matrix error can still miss the one row that matters for token retrieval.

## Motivation

Kernelized attention replaced the `T x T` score matrix with associative feature summaries in Task 6. Low-rank methods attack the same sequence bottleneck from a different angle: instead of changing the kernel, they assume the sequence interaction matrix itself has redundant structure and can be represented with fewer sequence degrees of freedom. The practical question is not whether this is sometimes true, but when it is true enough for a given task [@linformer_2020; @nystromformer_2021].

## Hypothesis

Smooth attention patterns should admit useful low-rank or landmark approximations, so reconstruction error should improve as rank or landmark count increases. Exact lookup-style behavior is more brittle: a rank budget can yield a modest Frobenius error while still changing the decisive token for one critical query.

## Baseline

The baseline is the uncompressed attention-like matrix or the exact softmax attention output. We compare three approximation families against that baseline: truncated SVD as the best rank-`k` reference by Frobenius norm, a Linformer-style sequence projection using deterministic mean pooling, and a small Nyström-style landmark reconstruction.

## Metric

The main global metric is relative Frobenius error, `||A - A_hat||_F / ||A||_F`. For projected sequence attention we also measure relative output error versus exact attention. For the retrieval failure case we add a task metric: whether the critical query still retrieves the correct token index after approximation.

## Mathematical derivation

For a matrix `A in R^{T x T}` with singular value decomposition `A = U Sigma V^T`, the Eckart-Young theorem states that the best rank-`k` approximation in Frobenius norm is

$$A_k = U_k \Sigma_k V_k^T, \qquad ||A - A_k||_F^2 = \sum_{r > k} \sigma_r^2.$$

That makes truncated SVD the right diagnostic reference: if even the best rank-`k` approximation is poor, learned low-rank attention will have a hard time too.

A Linformer-style projection assumes the sequence dimension can be compressed before attention:

$$\operatorname{softmax}(QK^T / \sqrt{d})V \approx \operatorname{softmax}(Q(EK)^T / \sqrt{d})(FV),$$

with `E, F in R^{k x T}` and `k << T`.

A Nyström-style landmark approximation samples landmark indices `L` and reconstructs a square matrix from sampled columns, the sampled intersection, and sampled rows:

$$A \approx A_{:,L} A_{L,L}^+ A_{L,:}. $$

The attraction is again sequence compression, but the risk is that global error can look acceptable while one critical row changes meaning [@linformer_2020; @nystromformer_2021].

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.attention_math import exact_attention
from llm_from_scratch.research.low_rank_attention import (
    frobenius_relative_error,
    make_mean_pool_projection,
    nystrom_matrix_approximation,
    projected_sequence_attention,
    svd_diagnostics,
    truncated_svd_approximation,
)

torch.manual_seed(0)
torch.set_printoptions(precision=5, sci_mode=False)

## PyTorch implementation

The helper module exposes the minimal research surface for this notebook: truncated SVD diagnostics, a deterministic mean-pool sequence projection for a Linformer-style toy path, and a Nyström-style landmark reconstruction. The code below builds two controlled matrices: one smooth matrix that should compress reasonably well, and one near-smooth matrix with a single rare retrieval row.

In [ ]:
def gaussian_smoothing_weights(length: int, sigma: float) -> torch.Tensor:
    positions = torch.arange(length, dtype=torch.float64)
    distances = positions[:, None] - positions[None, :]
    logits = -(distances.square()) / (2.0 * sigma * sigma)
    return torch.softmax(logits, dim=-1)

def row_stochastic(matrix: torch.Tensor) -> torch.Tensor:
    clipped = matrix.clamp_min(0.0)
    denom = clipped.sum(dim=-1, keepdim=True).clamp_min(torch.finfo(matrix.dtype).tiny)
    return clipped / denom

smooth_weights = gaussian_smoothing_weights(length=24, sigma=1.8)
retrieval_weights = smooth_weights.clone()
critical_query = 20
target_key = 4
retrieval_row = retrieval_weights[critical_query].clone()
retrieval_row[target_key] += 0.25
retrieval_row /= retrieval_row.sum()
retrieval_weights[critical_query] = retrieval_row

smooth_diag = svd_diagnostics(smooth_weights)
retrieval_diag = svd_diagnostics(retrieval_weights)

positions = torch.linspace(-1.0, 1.0, steps=12, dtype=torch.float64)
base_features = torch.stack([positions, positions.square(), torch.cos(positions)], dim=-1)
q = (0.5 * base_features).unsqueeze(0)
k = q.clone()
v = torch.stack([positions, torch.sin(positions)], dim=-1).unsqueeze(0)
exact_output, exact_weights = exact_attention(q, k, v)

{
    'smooth_top_singular_values': smooth_diag.singular_values[:6],
    'smooth_cumulative_energy': smooth_diag.cumulative_energy[:6],
    'retrieval_top_singular_values': retrieval_diag.singular_values[:6],
    'retrieval_cumulative_energy': retrieval_diag.cumulative_energy[:6],
    'critical_query_exact_top_key': int(retrieval_weights[critical_query].argmax().item()),
}

## Numerical checks

First verify the monotonic story on controlled cases. For the deterministic mean-pool projection, use divisor-aligned projected lengths so each pooled bucket has uniform width. Then test the retrieval failure case explicitly: a globally decent approximation is still unacceptable if the critical query retrieves the wrong token.

In [ ]:
rank_sweep = [2, 4, 8, 12, 24]
smooth_rank_errors = [
    frobenius_relative_error(smooth_weights, truncated_svd_approximation(smooth_weights, rank))
    for rank in rank_sweep
]
assert all(later < earlier for earlier, later in zip(smooth_rank_errors, smooth_rank_errors[1:]))

projection_sweep = [3, 4, 6, 12]
projected_output_errors = []
for projected_length in projection_sweep:
    projection = make_mean_pool_projection(12, projected_length, dtype=torch.float64)
    projected_output, projected_weights = projected_sequence_attention(q, k, v, projection=projection)
    projected_output_errors.append({
        'projected_length': projected_length,
        'relative_output_error': round(
            frobenius_relative_error(exact_output, projected_output),
            6,
        ),
        'projected_weight_shape': tuple(projected_weights.shape),
    })
assert all(
    later['relative_output_error'] < earlier['relative_output_error']
    for earlier, later in zip(projected_output_errors, projected_output_errors[1:])
)

landmark_sweep = [2, 4, 6, 8, 12, 24]
nystrom_errors = [
    {
        'num_landmarks': num_landmarks,
        'relative_fro_error': round(
            frobenius_relative_error(
                smooth_weights,
                nystrom_matrix_approximation(smooth_weights, num_landmarks=num_landmarks),
            ),
            6,
        ),
    }
    for num_landmarks in landmark_sweep
]
assert all(
    later['relative_fro_error'] < earlier['relative_fro_error']
    for earlier, later in zip(nystrom_errors, nystrom_errors[1:])
)

retrieval_rank = 8
retrieval_approx = row_stochastic(
    truncated_svd_approximation(retrieval_weights, rank=retrieval_rank)
)
retrieval_error = frobenius_relative_error(retrieval_weights, retrieval_approx)
exact_top = int(retrieval_weights[critical_query].argmax().item())
approx_top = int(retrieval_approx[critical_query].argmax().item())
assert retrieval_error < 0.15
assert approx_top != exact_top

{
    'smooth_rank_errors': [round(value, 6) for value in smooth_rank_errors],
    'projected_output_errors': projected_output_errors,
    'nystrom_errors': nystrom_errors,
    'retrieval_failure_case': {
        'critical_query': critical_query,
        'target_key': target_key,
        'relative_fro_error': round(retrieval_error, 6),
        'exact_top_key': exact_top,
        'approx_top_key': approx_top,
        'exact_target_weight': round(float(retrieval_weights[critical_query, target_key]), 6),
        'approx_target_weight': round(float(retrieval_approx[critical_query, target_key]), 6),
        'approx_local_weight': round(float(retrieval_approx[critical_query, critical_query]), 6),
    },
}

## Ablations

Two ablations matter here. First, increasing SVD rank or landmark count improves the smooth-matrix reconstruction, which supports the low-rank assumption on that toy regime. Second, the retrieval case shows a different axis of failure: even after the approximation budget is high enough to keep global error moderate, the critical row still shifts its top token from the distant key back to the local token. That is exactly the mismatch between global approximation quality and task preservation that low-rank attention papers need to stress-test.

In [ ]:
smooth_rank8 = truncated_svd_approximation(smooth_weights, rank=8)
retrieval_rank8 = row_stochastic(truncated_svd_approximation(retrieval_weights, rank=8))

{
    'smooth_rank8_error': round(frobenius_relative_error(smooth_weights, smooth_rank8), 6),
    'retrieval_rank8_error': round(frobenius_relative_error(retrieval_weights, retrieval_rank8), 6),
    'smooth_row_20_top_key': int(smooth_weights[critical_query].argmax().item()),
    'retrieval_row_20_exact_top_key': int(retrieval_weights[critical_query].argmax().item()),
    'retrieval_row_20_approx_top_key': int(retrieval_rank8[critical_query].argmax().item()),
}

## Assumptions

The notebook uses deterministic synthetic matrices, so it assumes these toys reflect real structural differences between smooth averaging and brittle retrieval. The mean-pool projection is also a fixed projection, not a learned one, which makes it a conservative Linformer-style sanity check rather than a competitive implementation. Because the pooling is hard-partitioned, uneven bucket boundaries can introduce discrete artifacts, so the projection sweep sticks to divisor-aligned budgets for cleaner comparisons.

## Risks

A good low-rank toy result does not imply that a trained model will remain stable layer after layer. Nyström-style reconstructions can also become numerically fragile when the landmark intersection is ill-conditioned. Finally, global error metrics can hide task failures unless the evaluation includes row-level or token-level checks.

## Failure criteria

Treat this notebook as failed if rank or landmark sweeps do not improve the smooth controlled cases, if the projected attention path does not approach the exact output as the projected sequence budget grows, or if the retrieval example cannot produce a wrong token under a still-moderate Frobenius error. Any of those outcomes would mean the toy setup is not separating approximation quality from task behavior clearly enough.

## Exercises

1. Re-derive the Eckart-Young residual formula and explain why it makes truncated SVD a useful diagnostic baseline.
2. Why does increasing projected sequence length help the deterministic Linformer-style toy even though the projection itself is fixed?
3. In the retrieval failure case, which metric catches the real error that Frobenius norm misses?

Companion exercise files:

- `exercises/research/17_low_rank_attention.md`
- `exercises/research/solutions/17_low_rank_attention_solutions.md`

## References

- Wang et al., *Linformer: Self-Attention with Linear Complexity* [@linformer_2020].
- Xiong et al., *Nyströmformer: A Nyström-Based Algorithm for Approximating Self-Attention* [@nystromformer_2021].